# Week 81, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w081d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import math
values = sorted([30, 10, 20, 40, 1000])
print(values[math.ceil(0.5 * len(values)) - 1], values[math.ceil(0.99 * len(values)) - 1])

**Predict before you run:** what will this print?



In [ ]:
import math
rate, latency = 2000, 0.025
print(rate * latency, math.ceil(rate * latency / 0.7))

**Predict before you run:** what will this print?



In [ ]:
from collections import OrderedDict
c = OrderedDict()
for k in "abcab":
    c[k] = True
    c.move_to_end(k)
print(list(c), c.popitem(last=False))

**Predict before you run:** what will this print?



In [ ]:
import zlib
print([zlib.crc32(k.encode()) % 2 == zlib.crc32(k.encode()) % 2 for k in ("x", "y")], hash("x") == hash("x"))

**Predict before you run:** what will this print?



In [ ]:
import heapq
a, b = [9, 5, 2], [8, 7, 1]
print(list(heapq.merge(a, b, reverse=True))[:4])

### Exercise: The cost of fan-out

A social service has accounts with these follower counts. Write `daily_writes(follower_counts, posts_per_day)`: the feed writes per day if every post is copied into every follower's feed (fan-out on write: each post costs one write per follower). Then `split_writes(follower_counts, posts_per_day, threshold)`: `(pushed, pulled_accounts)`, where `pushed` is the daily feed writes counting only accounts with at most `threshold` followers (the others are read at feed time instead), and `pulled_accounts` is how many accounts are above the threshold. `follower_counts` and `posts_per_day` are lists of the same length, one entry per account.

In [ ]:
def daily_writes(follower_counts, posts_per_day):
    ___

def split_writes(follower_counts, posts_per_day, threshold):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fanout():\n    \"celebrities dominate\"\n    followers = [120, 300, 45, 10_000_000, 80, 2_500_000]\n    posts = [3, 1, 5, 4, 2, 10]\n    assert daily_writes(followers, posts) == 65_001_045\n    assert split_writes(followers, posts, 10_000) == (1_045, 2)\n    assert split_writes(followers, posts, 100_000_000) == (65_001_045, 0)")

### Exercise: LRU with expiry

Write a class `ExpiringLRU(capacity, ttl, clock)` combining Tuesday's two caches: `put(key, value)` stores the value with the time it was stored (most recently used; evict the least recently used beyond `capacity`), and `get(key)` returns the value if present and younger than `ttl` (marking it used), or `None` (removing it if it has expired). Count `hits` and `misses`.

In [ ]:
from collections import OrderedDict

class ExpiringLRU:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"both limits\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    c = ExpiringLRU(2, 10, clock)\n    c.put(\"a\", 1)\n    c.put(\"b\", 2)\n    assert c.get(\"a\") == 1\n    c.put(\"c\", 3)\n    assert c.get(\"b\") is None and c.get(\"c\") == 3\n    clock.t = 10\n    assert c.get(\"a\") is None and c.get(\"c\") is None and (c.hits, c.misses) == (2, 3)")

### Exercise: Forge: a news feed

Write `FeedService` as described above.

In [ ]:
import heapq
from collections import deque

class FeedService:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _service(threshold=2):\n    class Clock:\n        t = 1000\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    return FeedService(clock, celebrity_threshold=threshold, feed_size=5, cache_ttl=30), clock\n\ndef ids(page):\n    return [p[\"id\"] for p in page[\"posts\"]]\n\ndef test_push():\n    \"ordinary accounts are pushed through the queue\"\n    f, clock = _service()\n    f.follow(\"ada\", \"bob\")\n    f.follow(\"cy\", \"bob\")\n    p1 = f.post(\"bob\", \"hello\")\n    assert ids(f.feed(\"ada\")) == [] and f.process_fanout() == 2\n    clock.t += 31\n    assert f.feed(\"ada\") == {\"posts\": [{\"id\": p1, \"author\": \"bob\", \"text\": \"hello\"}], \"next\": None}\n    for n in range(6):\n        f.post(\"bob\", f\"post {n}\")\n    assert f.process_fanout(batch=4) == 8 and f.process_fanout() == 4\n    clock.t += 31\n    assert ids(f.feed(\"ada\")) == [7, 6, 5, 4, 3] and ids(f.feed(\"ada\", limit=2)) == [7, 6]\n    assert f.feed(\"ada\", limit=2)[\"next\"] == 6 and ids(f.feed(\"ada\", limit=2, before=6)) == [5, 4]\n    try:\n        f.follow(\"ada\", \"ada\")\n    except ValueError as e:\n        assert str(e) == \"you can't follow yourself\"\n\ndef test_celebrities_are_pulled():\n    \"no fan-out for huge accounts; merged at read time\"\n    f, clock = _service(threshold=2)\n    for fan in (\"ada\", \"bob\", \"cy\"):\n        f.follow(fan, \"star\")\n    f.follow(\"ada\", \"dan\")\n    s1 = f.post(\"star\", \"big news\")\n    d1 = f.post(\"dan\", \"small news\")\n    s2 = f.post(\"star\", \"more news\")\n    assert f.process_fanout() == 1\n    assert ids(f.feed(\"ada\")) == [s2, d1, s1] and ids(f.feed(\"bob\")) == [s2, s1]\n    f.unfollow(\"ada\", \"star\")\n    assert ids(f.feed(\"ada\")) == [d1]\n    f.unfollow(\"ada\", \"dan\")\n    assert ids(f.feed(\"ada\")) == []\n\ndef test_cache():\n    \"short-lived cached reads\"\n    f, clock = _service()\n    f.follow(\"ada\", \"bob\")\n    f.post(\"bob\", \"one\")\n    f.process_fanout()\n    first = f.feed(\"ada\")\n    assert f.feed(\"ada\") == first and (f.cache_hits, f.cache_misses) == (1, 1)\n    f.post(\"bob\", \"two\")\n    f.process_fanout()\n    assert ids(f.feed(\"ada\")) == [1] and f.cache_hits == 2\n    clock.t += 30\n    assert ids(f.feed(\"ada\")) == [2, 1] and f.cache_misses == 2\n    f.follow(\"ada\", \"cy\")\n    f.feed(\"ada\")\n    assert f.cache_misses == 3, \"following clears the cached feed\"")

> **Where each week's idea appeared.** Look back at your feed: Monday's arithmetic (the mixed-review fan-out costs) decided the threshold; Tuesday's cache sits in front of `feed`, and its short TTL is why a new post can take up to 30 seconds to appear (is that acceptable here? for a bank statement it wouldn't be); Wednesday's queue keeps `post` fast and lets fan-out fall behind under load without anyone noticing; Thursday's sharding would split `feeds` by user id, and the celebrity posts would be read from wherever their author's shard is. Which of these would you change first if posting became slow? Which if reading did?

### Exercise: Forge bonus: a gateway that sheds load

Write a class `Gateway(clock, max_in_flight, per_user_rate, burst)`. `begin(user, kind)` decides whether a request may start, returning `(status, reason)`: requests of kind `"write"` or `"read"` (anything else is `(400, "unknown kind")`). A per-user token bucket (capacity `burst`, refilling at `per_user_rate` tokens a second, starting full) gives `(429, "rate limited")` when empty. If `in_flight` has reached `max_in_flight`, the request is shed: `(503, "overloaded")`. Also, once in-flight reaches 80% of `max_in_flight` (rounded down, at least 1), **reads** are shed with `(503, "shedding reads")` so writes, which matter more, still get through. Otherwise count it in flight and return `(200, "ok")`. Check in this order: kind, rate limit, full, shedding (a refused request doesn't use a token unless it was refused only for load). `end()` finishes one request. `stats()` returns a dict counting each status (as keys `200`, `429`, `503`, `400`, sorted, only those seen) plus `"in_flight"`.

In [ ]:
class Gateway:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gateway():\n    \"rate limits, full and shedding\"\n    class Clock:\n        t = 0.0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    g = Gateway(clock, max_in_flight=5, per_user_rate=1.0, burst=2)\n    assert g.begin(\"ada\", \"read\") == (200, \"ok\") and g.begin(\"ada\", \"read\") == (200, \"ok\")\n    assert g.begin(\"ada\", \"read\") == (429, \"rate limited\")\n    assert g.begin(\"bob\", \"write\") == (200, \"ok\") and g.begin(\"cy\", \"read\") == (200, \"ok\")\n    assert g.begin(\"dan\", \"read\") == (503, \"shedding reads\") and g.begin(\"dan\", \"write\") == (200, \"ok\")\n    assert g.begin(\"eve\", \"write\") == (503, \"overloaded\") and g.begin(\"eve\", \"delete\") == (400, \"unknown kind\")\n    g.end()\n    g.end()\n    assert g.begin(\"eve\", \"read\") == (200, \"ok\")\n    clock.t = 1.0\n    assert g.begin(\"ada\", \"read\") == (503, \"shedding reads\")\n    assert g.stats() == {200: 6, 400: 1, 429: 1, 503: 3, \"in_flight\": 4}\n\ndef test_small_limits():\n    \"at least one slot before shedding\"\n    class Clock:\n        def __call__(self):\n            return 0.0\n    g = Gateway(Clock(), max_in_flight=1, per_user_rate=10, burst=10)\n    assert g.begin(\"a\", \"read\") == (200, \"ok\") and g.begin(\"b\", \"read\") == (503, \"overloaded\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a news feed

A feed shows each user the newest posts from everyone they follow. Two ways to build it:

- **Fan-out on write (push)**: when someone posts, copy the post's id into the stored feed of each follower. Reading a feed is then one quick lookup. But a post by an account with ten million followers means ten million writes.
- **Fan-out on read (pull)**: store nothing per reader; at read time, fetch the recent posts of everyone the reader follows and merge them. Posting is cheap, but every feed read does a lot of work.

Large feeds use **both**: push for ordinary accounts, pull for the few with huge followings ("celebrities"), merged at read time. The push work goes through a **queue**, so posting stays fast.

Build `FeedService(clock, celebrity_threshold=1000, feed_size=200, cache_ttl=30)`:

- `follow(follower, followee)` and `unfollow(follower, followee)`; following yourself raises `ValueError("you can't follow yourself")`. Following or unfollowing clears the follower's cached feeds.
- `post(author, text)` stores the post with the next id (from 1) and the current time, queues a fan-out job, and returns the id.
- `process_fanout(batch=100)` handles up to `batch` queued jobs: for an author with **at most** `celebrity_threshold` followers, add the post id to each follower's stored feed (newest first, keeping only `feed_size` ids). Authors above the threshold are skipped (they'll be pulled). It returns the number of feed writes made.
- `feed(user, limit=10, before=None)` returns `{"posts": [...], "next": CURSOR}`: posts (each `{"id", "author", "text"}`) newest first by id, drawn from the user's stored feed **plus** the posts of every celebrity they currently follow (above the threshold now), excluding authors they no longer follow, only ids **below** `before` if it's given, at most `limit` of them; `next` is the last post's id if there might be more (a full page), else `None`. Results are cached per `(user, limit, before)` for `cache_ttl` seconds; count `cache_hits` and `cache_misses`.

## Forge bonus: a gateway that sheds load

When traffic exceeds what a service can handle, it's far better to refuse some requests quickly than to accept them all and answer everyone slowly. A **gateway** in front of the service does that: per-user rate limits stop one client hogging capacity, a **concurrency limit** caps the requests in progress, and when the service is overloaded it **sheds** the least important work first.

## Weekly quiz

## Week 81 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can measure latency with percentiles, estimate load, apply Little's law, balance load and find bottlenecks. ([Day 1](w081d1.html))
- I can build LRU and TTL caches, keep them honest with invalidation, stop stampedes and design safe keys. ([Day 2](w081d2.html))
- I can use bounded queues, acknowledgements, dead letters, idempotent consumers, pub/sub and partitioned logs. ([Day 3](w081d3.html))
- I can reason about replication lag, failover, partitioning, consistent hashing, scatter-gather and quorums. ([Day 4](w081d4.html))
- I can design a URL shortener from requirements and estimates to ids, redirects, caching, click counts and rate limits. ([Day 5](w081d5.html))
- I built a news feed with push and pull fan-out, and a gateway that sheds load. (Today)

Tomorrow is a **rest day**. Next week is **System design II**: distributed systems concepts, observability, reliability, and design interview practice.